# PP-OCRv6 price recognition

Install with `python -m pip install -e ".[notebooks,ppocr,ppocr-tensorrt]"` from the repository root. Open this notebook in that environment. The demo uses fictional price crops and downloads official pretrained weights. It checks the workflow rather than estimating accuracy on real price tags.

## 1. Configure the run

The demo runs one epoch. To train on your own data, set `DATASET` to a folder with `train` and `val` subfolders containing image and same-stem UTF-8 TXT pairs, and increase `EPOCHS`. Set the run switches to True when ready.

In [ ]:
from pathlib import Path
import subprocess
import sys
import yaml

ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / 'configs/ppocr_price.yaml').exists())
DATASET = ROOT / 'dummy_data/prices'
EPOCHS = 1
RUN_TRAINING = False
RUN_TENSORRT = False
WORK = ROOT / 'artifacts/notebook_price'
WORK.mkdir(parents=True, exist_ok=True)
cfg = yaml.safe_load((ROOT / 'configs/ppocr_price.yaml').read_text())
cfg.update(dataset_root=str(DATASET.resolve()), output_dir=str(WORK), epochs=EPOCHS, batch_size=4, auxiliary=False)
config = WORK / 'config.yaml'
config.write_text(yaml.safe_dump(cfg))

def run(command, *args):
    subprocess.run([sys.executable, '-m', 'parseq_ocr.ppocr_cli', command, '--config', str(config), *args], cwd=ROOT, check=True)

example = next(p for p in sorted((DATASET / 'val').iterdir()) if p.suffix.lower() in {'.png', '.jpg', '.jpeg'})
print('Dataset:', DATASET, '| Example:', example.name)


## 2. Prepare weights and train

In [ ]:
if RUN_TRAINING:
    run('prepare')
    run('train')

## 3. Check native inference and exact-match accuracy

In [ ]:
if RUN_TRAINING:
    run('infer', '--image', str(example))
    run('evaluate')

## 4. Export, build, and run TensorRT

Run this after training or with an existing checkpoint in the configured output directory. Build the engine on the GPU you will use for inference.

In [ ]:
if RUN_TENSORRT:
    run('export')
    run('build')
    run('infer', '--runtime', 'tensorrt', '--image', str(example))
    run('evaluate', '--runtime', 'tensorrt')
    run('benchmark', '--runtime', 'tensorrt')